# 01 — Tokenizer Setup

One-time setup: extracts `vocab.json` and `merges.txt` from the bundled BPE model file
into `artifacts/tokenizers/tok_bpe_8k/`, then verifies the tokenizer loads correctly.

Run this once before `02_dataset.ipynb`.

In [ ]:
# Cell 1 — project root
# Change this path to match your environment. This is the ONLY cell that differs
# between local and cloud.
import os

# os.chdir("/notebooks/alg3")                        # Nebius / RunPod / cloud
os.chdir("/Users/dmitrit/Documents/dev/alg3")       # local

print("Working directory:", os.getcwd())

In [ ]:
# Cell 2 — install dependencies
# Skip if already installed in your environment.
# For NVIDIA GPU cloud: torch is usually pre-installed with CUDA support.
# !pip install torch tokenizers tensorboard numpy --quiet

In [ ]:
# Cell 3 — extract tokenizer files
# Reads  data/raw/98m_8k_bpe.model  and  data/raw/98m_8k_bpe.vocab
# Writes artifacts/tokenizers/tok_bpe_8k/vocab.json + merges.txt
import json, pathlib

model_path = pathlib.Path("data/raw/98m_8k_bpe.model")
vocab_path  = pathlib.Path("data/raw/98m_8k_bpe.vocab")
out_dir     = pathlib.Path("artifacts/tokenizers/tok_bpe_8k")
out_dir.mkdir(parents=True, exist_ok=True)

model_data = json.loads(model_path.read_text(encoding="utf-8"))
merges_txt = model_data["merges"]

vocab = {}
with open(vocab_path, encoding="utf-8") as f:
    for idx, line in enumerate(f):
        parts = line.strip().split("\t")
        if parts:
            vocab[parts[0]] = idx

(out_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False), encoding="utf-8")
(out_dir / "merges.txt").write_text(merges_txt, encoding="utf-8")

print(f"vocab.json : {len(vocab)} tokens -> {out_dir / 'vocab.json'}")
print(f"merges.txt : written           -> {out_dir / 'merges.txt'}")

In [ ]:
# Cell 4 — verify tokenizer loads and encodes correctly
import sys
sys.path.insert(0, "src")

from tokenizer import Tokenizer

tok = Tokenizer.load("tok_bpe_8k")
print(f"Loaded tok_id : {tok.tok_id}")
print(f"Vocab size    : {tok.vocab_size}")

sample = "Once upon a time, there was a little girl named Lily."
ids    = tok.encode(sample)
decoded = tok.decode(ids)

print(f"\nSample text : {sample!r}")
print(f"Token IDs   : {ids}")
print(f"Decoded     : {decoded!r}")